# Lab07.1 — From Image Pixels to an FPGA Data Stream

**Platform:** Digilent Zybo legacy (Zynq-7010), PYNQ 3.0.1.  
**Prerequisites:** Lab04 image basics, Lab06 PS–PL communication.  
**Hardware required in this notebook:** **none** — `lab07.bit` and Vivado are **not** required.

## Why are we doing this?

In Lab04, Python/OpenCV operated on complete images. In Lab06, the ARM wrote a few operands to AXI registers to control hardware. In Lab07 we want to pass **an entire image** from ARM memory to an FPGA image-processing circuit using **AXI DMA**. Before doing that in Lab07.2, we need to understand how image coordinates, pixel intensity, thresholding, and AXI-Stream data words relate.

At the end of this notebook, you will be able to:

1. Read pixel coordinates from an image annotated with a **32-pixel grid**.
2. Select a rectangular **region of interest (ROI)** and derive its exact pixel bounds.
3. Explain what a grayscale threshold does, using the image histogram and visual masks.
4. Pack four 8-bit pixels into a 32-bit AXI-Stream word and predict the FPGA output.
5. Distinguish **ideal FPGA processing throughput** from **real end-to-end DMA execution time**.

The image is synthetic so that results are reproducible. All computation here is **software simulation/reference computation**; actual PL processing begins in Lab07.2.

## Part A — Understanding the image and its coordinates

### Step 1 — Load the test image (observe a 2-D array, not just a picture)

An 8-bit grayscale image contains **one number per pixel**, between 0 (black) and 255 (white). Intermediate values are shades of gray. The Python array is indexed as `image[y, x]`: **row (y) first, column (x) second**.

For a 256 × 256 image, valid pixel indices are **0 through 255** on both axes. The top-left pixel is `(x=0, y=0)`. The `y` coordinate increases **downward**, not upward as in an ordinary mathematical graph.

The cell below reads `images/scene_256.png`, verifies the data type, prints some properties, and displays **pixel coordinate axes** with a grayscale colorbar. Keep the extracted `Lab07/` folder intact: the notebook belongs in `Lab07/notebooks/`, the image in `Lab07/images/`.

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Locate the existing image relative to the Jupyter working directory.
# Normal layout: Lab07/notebooks/ and Lab07/images/ are sibling folders.
candidates = [Path('../images/scene_256.png'),
              Path('images/scene_256.png'),
              Path('scene_256.png')]
image_path = next((p for p in candidates if p.is_file()), None)
if image_path is None:
    raise FileNotFoundError(
        'Missing scene_256.png. Preserve Lab07/notebooks/ and Lab07/images/ '
        'when extracting the ZIP. Current directory: ' + str(Path.cwd()))

gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise RuntimeError('OpenCV could not read ' + str(image_path))
H, W = gray.shape
assert gray.dtype == np.uint8
print('File:', image_path.resolve())
print('Image size: {} columns × {} rows'.format(W, H))
print('NumPy array shape (rows, columns):', gray.shape)
print('Data type:', gray.dtype, '| min:', int(gray.min()), '| max:', int(gray.max()))
print('Example: image[0,0] =', int(gray[0, 0]))

fig, ax = plt.subplots(figsize=(7, 6))
view = ax.imshow(gray, cmap='gray', vmin=0, vmax=255,
                 extent=(0, W, H, 0), interpolation='nearest')
ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_xticks(np.arange(0, W+1, 32))
ax.set_yticks(np.arange(0, H+1, 32))
ax.set_xlabel('x = image column (pixel coordinate)')
ax.set_ylabel('y = image row (pixel coordinate)')
ax.set_title('Original 256 × 256 grayscale image — coordinates visible')
fig.colorbar(view, ax=ax, label='Pixel intensity (0 black → 255 white)', shrink=.82)
plt.tight_layout(); plt.show()

### Step 2 — Understand the 32 × 32 pixel grid

We overlay an **8 × 8 grid** on the 256 × 256 image. Each cell has a width and height of 32 pixels.

**There are two coordinate systems:**

- **Pixel coordinates:** `x=0…255` and `y=0…255`. Axis markers at `0, 32, 64, …, 256` mark **cell boundaries**. The label `256` is the boundary *after* the last pixel, not a valid pixel index.
- **Grid-cell indices:** columns **C0…C7** from left to right, and rows **R0…R7** from top to bottom. For example, **C4** covers pixel x coordinates **128…159**, and **R2** covers y coordinates **64…95**.

The next diagram draws the grid directly over the original grayscale image. The **top axis** names grid columns, the **right axis** names grid rows, while the bottom/left axes display exact pixel boundaries.

**Student task:** before running the next cell, identify the grid cells containing the bright circle and the dark circular hole.

In [ ]:
CELL = 32
assert (W, H) == (256, 256) and W % CELL == 0 and H % CELL == 0

fig, ax = plt.subplots(figsize=(9, 7.2))
ax.imshow(gray, cmap='gray', vmin=0, vmax=255,
          extent=(0, W, H, 0), interpolation='nearest')
ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_aspect('auto')  # twiny+twinx need adjustable box/auto
for x in range(0, W+1, CELL):
    ax.axvline(x, color='#f4a900', linewidth=1.1)
for y in range(0, H+1, CELL):
    ax.axhline(y, color='#f4a900', linewidth=1.1)
ax.set_xticks(np.arange(0, W+1, CELL))
ax.set_yticks(np.arange(0, H+1, CELL))
ax.set_xlabel('x (pixel boundary)')
ax.set_ylabel('y (pixel boundary, increases downward)')
ax.set_title('Image grid: pixels on lower/left axes, cell indices on upper/right axes')

# A second set of axes shows grid-cell indices at CELL CENTERS.
col_ax = ax.twiny()
col_ax.set_xlim(0, W)
col_ax.set_xticks(np.arange(CELL/2, W, CELL))
col_ax.set_xticklabels(['C{}'.format(k) for k in range(W//CELL)])
col_ax.set_xlabel('Grid columns')

row_ax = ax.twinx()
row_ax.set_ylim(H, 0)
row_ax.set_yticks(np.arange(CELL/2, H, CELL))
row_ax.set_yticklabels(['R{}'.format(k) for k in range(H//CELL)])
row_ax.set_ylabel('Grid rows')

plt.tight_layout(); plt.show()
print('Example: C4 -> x=128..159; R2 -> y=64..95')

### Step 3 — Select the ROI, and **show exactly where it comes from**

A **region of interest (ROI)** is a selected rectangular part of an image. Here we select a **3 × 3 group of grid cells** that contains most of the bright circle. The **object (circle)** and the **ROI (rectangle)** are not the same thing: our ROI still includes some background around the object.

The selected grid cells are:

- **Columns C4, C5, C6** → indices `4, 5, 6`.
- **Rows R2, R3, R4** → indices `2, 3, 4`.

In Python, slices include the starting position but **exclude the ending position**. Thus `gray[64:160, 128:224]` includes the pixel indices **x = 128…223** and **y = 64…159**.

Derivation:

\[
\begin{aligned}
x_0 &= 4\times32=128, &x_{stop}&=7\times32=224,\\
y_0 &= 2\times32=64, &y_{stop}&=5\times32=160,\\
W_{ROI}&=224-128=96, &H_{ROI}&=160-64=96.
\end{aligned}
\]

**Important:** the plot will show the ROI as a **red rectangle** on top of the original image, so the selected coordinates are visually verifiable. Change the row/column parameters to choose another ROI; all results will update automatically.

In [ ]:
# These four values are the ONLY variables needed to choose a new ROI.
column_start = 4       # first INCLUDED grid column
column_stop = 7        # first EXCLUDED grid column
row_start = 2          # first INCLUDED grid row
row_stop = 5           # first EXCLUDED grid row

assert 0 <= column_start < column_stop <= W//CELL
assert 0 <= row_start < row_stop <= H//CELL
x0, x_stop = column_start*CELL, column_stop*CELL
y0, y_stop = row_start*CELL, row_stop*CELL
roi = gray[y0:y_stop, x0:x_stop]   # NumPy: image[y, x]

print('Selected grid columns:', list(range(column_start, column_stop)))
print('Selected grid rows:   ', list(range(row_start, row_stop)))
print('Top-left pixel (x,y): ({}, {})'.format(x0, y0))
print('Bottom-right INCLUDED pixel (x,y): ({}, {})'.format(x_stop-1, y_stop-1))
print('Python crop: gray[{}:{}, {}:{}]'.format(y0, y_stop, x0, x_stop))
print('ROI shape (height, width):', roi.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
a = axes[0]
a.imshow(gray, cmap='gray', vmin=0, vmax=255,
         extent=(0, W, H, 0), interpolation='nearest')
for x in range(0, W+1, CELL):
    a.axvline(x, color='#f4a900', linewidth=.9)
for y in range(0, H+1, CELL):
    a.axhline(y, color='#f4a900', linewidth=.9)
a.add_patch(Rectangle((x0, y0), x_stop-x0, y_stop-y0,
                      facecolor='red', alpha=.12, edgecolor='none'))
a.add_patch(Rectangle((x0, y0), x_stop-x0, y_stop-y0,
                      fill=False, edgecolor='red', linewidth=3))
a.set_xlim(0, W); a.set_ylim(H, 0)
a.set_xticks(np.arange(0, W+1, CELL)); a.set_yticks(np.arange(0, H+1, CELL))
a.set_title('Original image: SELECTED ROI outlined in RED')
a.set_xlabel('x (pixels)'); a.set_ylabel('y (pixels)')

axes[1].imshow(roi, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
axes[1].set_title('Extracted ROI: {} × {} pixels'.format(roi.shape[1], roi.shape[0]))
axes[1].set_xlabel('x within ROI'); axes[1].set_ylabel('y within ROI')
plt.tight_layout(); plt.show()

### Step 4 — Verify a few pixel coordinates numerically

A pixel is a **single brightness value**, not an RGB triplet. The following cell samples explicit `(x, y)` positions and displays their coordinates on the image. This shows how the visible shapes correspond to values stored in the array.

Remember the indexing order: to read pixel `(x, y)` we must write **`gray[y, x]`**. This convention remains the same when moving an ROI or when flattening an image for DMA.

In [ ]:
sample_points = [
    ('dark background', 8, 8),
    ('inside triangle', 86, 90),
    ('bright circle', 170, 96),
    ('dark hole', 195, 203),
]
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(gray, cmap='gray', vmin=0, vmax=255,
          extent=(0, W, H, 0), interpolation='nearest')
for label, x, y in sample_points:
    intensity = int(gray[y, x])
    print('{:<18} at (x={:3d}, y={:3d}): intensity = {:3d}'.format(label, x, y, intensity))
    ax.plot(x+.5, y+.5, marker='o', markersize=6, color='red')
    ax.annotate('{}: {}'.format(label, intensity), (x+.5, y+.5),
                xytext=(8, -14), textcoords='offset points',
                fontsize=8, color='yellow', weight='bold')
ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_xticks(np.arange(0, W+1, 32)); ax.set_yticks(np.arange(0, H+1, 32))
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('A few explicit pixel positions and their grayscale values')
plt.tight_layout(); plt.show()

## Part B — From grayscale to a binary mask

### Step 5 — Understand the exact thresholding rule

The FPGA accelerator will apply this **identical rule independently to every pixel**:

\[
P_{out}=\begin{cases}255,&P_{in}\geq T\\0,&P_{in}<T.\end{cases}
\]

Here `T` is an **8-bit threshold** from 0 to 255. Pixels equal to the threshold **become white**. This detail matters: the standard OpenCV `cv2.THRESH_BINARY` comparison uses **strictly greater than** (`>`), not `>=`.

At `T = 128`:

- Input **127** → output **0** (black).
- Input **128** → output **255** (white).
- Input **129** → output **255** (white).

The next cell imports the **same reference implementation** used by Lab07.2, computes a mask for the entire image, and reports the percentage of white pixels. This is a software calculation, not a measurement of FPGA performance.

In [ ]:
import sys
# Our helper is in Lab07/tools; add its folder to Python's import path.
helper_candidates = [Path('../tools'), Path('tools')]
helper_dir = next((p for p in helper_candidates if (p/'dma_threshold.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Missing Lab07/tools/dma_threshold.py. Extract the complete Lab07 package.')
sys.path.insert(0, str(helper_dir.resolve()))
from dma_threshold import reference_threshold

THRESHOLD = 128
mask = reference_threshold(gray, THRESHOLD)
white_count = int(np.count_nonzero(mask))
white_percent = 100.0 * white_count / mask.size
print('Rule: pixel >= {} becomes 255, otherwise 0'.format(THRESHOLD))
print('Pixels evaluated: {} = {} × {}'.format(mask.size, H, W))
print('White pixels: {} ({:.1f}%)'.format(white_count, white_percent))
print('Unique mask intensities:', np.unique(mask).tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Input: grayscale (0...255)')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Binary mask: T={} | {:.1f}% white'.format(THRESHOLD, white_percent))
for ax in axes:
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)')
plt.tight_layout(); plt.show()

### Step 6 — Use the grayscale histogram to understand noisy segmentation

A **histogram** counts how many pixels have each possible intensity, 0 through 255. Thresholding at `T` makes the histogram portion at **T and above** white; everything below T becomes black.

The test image intentionally contains objects of different brightness and a textured/gradient background. This explains why the mask at 128 sometimes has **isolated white pixels near the right edge**: parts of the background cross the threshold because of the intensity gradient and noise. This is **not a DMA or FPGA error**.

The figure below shows the intensity distribution and three candidate threshold positions. Compare where `T=64`, `T=128` and `T=192` fall in the histogram.

In [ ]:
hist = np.bincount(gray.ravel(), minlength=256)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(np.arange(256), hist, width=1, color='#555555')
for t, color in [(64, '#1b9e77'), (128, '#d95f02'), (192, '#7570b3')]:
    ax.axvline(t, color=color, linewidth=2, linestyle='--', label='T = {}'.format(t))
ax.set_xlim(0, 255)
ax.set_xlabel('Grayscale pixel intensity')
ax.set_ylabel('Number of pixels')
ax.set_title('Image intensity histogram — threshold means splitting this distribution')
ax.grid(axis='y', alpha=.2)
ax.legend()
plt.tight_layout(); plt.show()
print('Pixels below 128:', int(np.count_nonzero(gray < 128)))
print('Pixels at or above 128:', int(np.count_nonzero(gray >= 128)))

### Step 7 — Compare three thresholds visually

Try three thresholds on **the same unchanged input image**. A low threshold selects more pixels, including the darker background. A high threshold leaves only the brightest objects.

- **T = 64:** a large part of the background becomes white; the shapes are harder to distinguish.
- **T = 128:** bright shapes are separated more clearly, but some background noise is still selected.
- **T = 192:** only especially bright regions remain, so dimmer objects disappear.

For every mask, the percentage printed above the image is calculated, not guessed. Ask yourself: **Can one global threshold isolate every object perfectly?**

In [ ]:
levels = (64, 128, 192)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
fractions = []
for ax, t in zip(axes, levels):
    candidate = reference_threshold(gray, t)
    white = 100.0*np.count_nonzero(candidate)/candidate.size
    fractions.append(white)
    ax.imshow(candidate, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    ax.set_title('Threshold = {}\nWhite pixels = {:.1f}%'.format(t, white))
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Identical input image — different decisions for each pixel', y=1.02)
plt.tight_layout(); plt.show()

for t, percentage in zip(levels, fractions):
    print('T = {:3d}  ->  {:5.1f}% of all pixels are white'.format(t, percentage))
assert fractions[0] >= fractions[1] >= fractions[2]

### Step 8 — An important comparison: `>=` versus OpenCV's `>`

This laboratory uses **`>=`** because that is what the FPGA RTL implements. However, OpenCV's `cv2.threshold(image, T, 255, cv2.THRESH_BINARY)` uses **`>`**. At precisely the threshold value the results differ.

This cell applies both comparisons to three deliberately chosen pixels. It also shows how to make OpenCV match the FPGA for an **integer threshold T ≥ 1**: pass `T-1` to `cv2.threshold` (because intensities are integers).

Why do this? Lab07.2 compares outputs **pixel by pixel**: using the wrong software reference would produce apparent hardware errors that are really just differences in the comparison operator.

In [ ]:
boundary_pixels = np.array([[127, 128, 129]], dtype=np.uint8)
fpga_reference = reference_threshold(boundary_pixels, 128)
_, opencv_default = cv2.threshold(boundary_pixels, 128, 255, cv2.THRESH_BINARY)
_, opencv_matching = cv2.threshold(boundary_pixels, 127, 255, cv2.THRESH_BINARY)
print('Input pixels:                      ', boundary_pixels.tolist()[0])
print('Our FPGA rule (pixel >= 128):     ', fpga_reference.tolist()[0])
print('OpenCV THRESH_BINARY at 128 (> 128):', opencv_default.tolist()[0])
print('OpenCV THRESH_BINARY at 127 (> 127):', opencv_matching.tolist()[0])
assert np.array_equal(fpga_reference, opencv_matching)
assert not np.array_equal(fpga_reference, opencv_default)

## Part C — How an image enters an FPGA accelerator

### Step 9 — Why four pixels fit into a 32-bit AXI-Stream word

Each grayscale pixel has **8 bits**. Our custom image threshold accelerator has a **32-bit AXI-Stream `TDATA` bus**, so it can carry **four consecutive pixels per accepted transfer** (*AXI beat*):

\[
32\ \text{bits per beat}\,/\,8\ \text{bits per pixel}=4\ \text{pixels per beat}.
\]

To make this visible, we choose **eight known pixel values**, rather than eight similar-valued background pixels. The values span **both sides of 128**, including 127 and 128, so the output contains both black and white pixels.

| Pixel | P0 | P1 | P2 | P3 | P4 | P5 | P6 | P7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Input intensity | 23 | 90 | 127 | 128 | 129 | 180 | 220 | 255 |
| Output at T=128 | 0 | 0 | 0 | 255 | 255 | 255 | 255 | 255 |

The next figure shows the eight input pixels as a grayscale strip and the eight output pixels as a black/white strip. The orange vertical divider separates the **two 32-bit AXI words**.

In [ ]:
AXI_THRESHOLD = 128
pixels = np.array([23, 90, 127, 128, 129, 180, 220, 255], dtype=np.uint8)
expected_pixels = reference_threshold(pixels.reshape(1, -1), AXI_THRESHOLD).reshape(-1)

fig, axes = plt.subplots(2, 1, figsize=(12, 3.5), sharex=True)
for ax, arr, title in [
    (axes[0], pixels, 'Eight INPUT pixels — grayscale intensity'),
    (axes[1], expected_pixels, 'Eight OUTPUT pixels — binary threshold at T=128')
]:
    ax.imshow(arr.reshape(1, -1), cmap='gray', vmin=0, vmax=255,
              interpolation='nearest', extent=(-.5, 7.5, .5, -.5), aspect='auto')
    for i, value in enumerate(arr):
        ink = 'black' if int(value) >= 135 else 'white'
        ax.text(i, 0, str(int(value)), ha='center', va='center',
                fontsize=12, color=ink, weight='bold')
    ax.axvline(3.5, color='#f4a900', linewidth=3)
    ax.set_title(title)
    ax.set_yticks([])
    ax.set_xlim(-.5, 7.5)
    ax.set_xticks(np.arange(8))
    ax.set_xticklabels(['P{}'.format(i) for i in range(8)])
axes[1].set_xlabel('AXI beat 0: pixels P0…P3          |          AXI beat 1: pixels P4…P7')
plt.tight_layout(); plt.show()
print('Input  pixels:', pixels.tolist())
print('Output pixels:', expected_pixels.tolist())

### Step 10 — Pack the four byte lanes and predict `TDATA`

The word is assembled from four 8-bit pixels **in the order shown below**:

| AXI `TDATA` bit range | 31–24 | 23–16 | 15–8 | 7–0 |
|---|---|---|---|---|
| Pixel index in the beat | P3 | P2 | P1 | P0 |
| First-beat decimal intensity | 128 | 127 | 90 | 23 |
| First-beat hexadecimal byte | `80` | `7F` | `5A` | `17` |

Therefore the first input word is **`0x807F5A17`**. Notice that hexadecimal numbers are printed with their **most significant byte on the left**, while the first transmitted image pixel occupies the **least significant byte on the right**. This is a property of our lane mapping; it is not a reversal of the image itself.

The general formula is:

\[
TDATA=P_0+(P_1\ll8)+(P_2\ll16)+(P_3\ll24).
\]

After thresholding, the first group `[23, 90, 127, 128]` becomes `[0, 0, 0, 255]`, so the output word is **`0xFF000000`**. The second input group becomes four white pixels, producing **`0xFFFFFFFF`**.

The Python code below packs pixels with bit shifts **explicitly**. It does **not access FPGA hardware**: we are predicting exactly what the streaming accelerator should output.

In [ ]:
def pack_four(byte_values):
    """Build one 32-bit word with pixel 0 in bits 7..0, then 1, 2, 3."""
    if len(byte_values) != 4:
        raise ValueError('Expected exactly four 8-bit pixels')
    return sum(int(v) << (8*i) for i, v in enumerate(byte_values))

input_words = [pack_four(pixels[k:k+4]) for k in (0, 4)]
output_words = [pack_four(expected_pixels[k:k+4]) for k in (0, 4)]
for beat in range(2):
    a, b = pixels[4*beat:4*beat+4], expected_pixels[4*beat:4*beat+4]
    print('Beat {}: IN {} -> TDATA 0x{:08X}'.format(beat, a.tolist(), input_words[beat]))
    print('        OUT {} -> TDATA 0x{:08X}'.format(b.tolist(), output_words[beat]))

assert input_words == [0x807F5A17, 0xFFDCB481]
assert output_words == [0xFF000000, 0xFFFFFFFF]
print('PASS: packing and threshold calculations agree with the expected words.')

### Step 11 — Learn the AXI-Stream handshake: `TDATA`, `TVALID`, `TREADY`, `TKEEP`, `TLAST`

A 32-bit data word is not enough: DMA must know **when the data is valid**, **when the receiver can accept it**, **which of the four bytes are meaningful**, and **where the transfer ends**.

- `TDATA[31:0]`: contains four 8-bit pixel lanes.
- `TVALID=1`: the sender is presenting a valid word.
- `TREADY=1`: the receiver is able to accept it.
- **A word is transferred only when `TVALID=1` and `TREADY=1` on a clock edge.** If the receiver asserts `TREADY=0`, the sender must hold its data stable until acceptance.
- `TKEEP=0b1111`: all four pixel bytes are valid. The current Lab07.2 experiment uses image widths divisible by four, so every beat has four valid bytes.
- `TLAST=1`: the **last word of the entire image transfer** (not automatically the last word in every image row). Our FPGA RTL forwards both `TKEEP` and `TLAST` unchanged; the S2MM DMA relies on `TLAST` to finish the frame.

For eight pixels, there are **two accepted beats**. Only the second one has `TLAST=1`. The figure below is a **conceptual protocol table**, not a hardware timing capture.

In [ ]:
handshake = [
    ('0', '1', '1', '0x807F5A17', '1111', '0', 'accepted: P0–P3'),
    ('1', '1', '1', '0xFFDCB481', '1111', '1', 'accepted: P4–P7 / end of frame'),
]
headers = ['Beat', 'TVALID', 'TREADY', 'Input TDATA', 'TKEEP', 'TLAST', 'Meaning']
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.axis('off')
table = ax.table(cellText=handshake, colLabels=headers, loc='center',
                 cellLoc='center', colLoc='center',
                 colWidths=[.07,.08,.09,.19,.10,.08,.39])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.0)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#d9eaf2')
    elif row == 2:
        cell.set_facecolor('#e9f5e9')
ax.set_title('Two AXI-Stream beats, no stalls (protocol illustration)', pad=12)
plt.tight_layout(); plt.show()
print('If TREADY = 0, the word is NOT accepted in that clock cycle.')
print('For N valid pixels divisible by 4: beats = N / 4; TLAST on final beat only.')

### Step 12 — Connect the image ROI to the streaming interface

Return to the **96 × 96 ROI** selected in Step 3. Before sending it to DMA, software would store its pixels as a contiguous byte sequence (typically **row by row**). The four first pixels from row 0 form beat 0; the next four form beat 1, and so on.

For our default ROI:

\[
96\times96=9216\ \mathrm{pixels},\qquad 9216/4=2304\ \mathrm{AXI\ beats}.
\]

The same principle applies to the complete **256 × 256** image: 65,536 pixels become 16,384 beats. The ROI selection itself is **not** done in the FPGA: in this laboratory, Python chooses the ROI, and the accelerator only applies thresholding to the transmitted pixel stream.

The next cell displays the first two input/output 32-bit words derived from the **actual selected ROI**, now that we have understood packing using our clear eight-pixel example.

In [ ]:
roi_flat = np.ascontiguousarray(roi).reshape(-1)
assert roi_flat.size % 4 == 0
roi_mask = reference_threshold(roi, AXI_THRESHOLD).reshape(-1)
print('ROI resolution: {} × {}'.format(roi.shape[1], roi.shape[0]))
print('Total ROI pixels:', roi_flat.size)
print('AXI-Stream words needed:', roi_flat.size // 4)
print('First 8 ROI pixels: ', roi_flat[:8].tolist())
print('First 8 outputs:    ', roi_mask[:8].tolist())
for beat in range(2):
    start = 4*beat
    word_in = pack_four(roi_flat[start:start+4])
    word_out = pack_four(roi_mask[start:start+4])
    print('Real ROI beat {}: input 0x{:08X} -> output 0x{:08X}'.format(beat, word_in, word_out))
print('Whole image: {} pixels -> {} 32-bit beats'.format(gray.size, gray.size // 4))

## Part D — Predicting FPGA performance before using DMA

### Step 13 — Calculate the **ideal steady-state streaming time** (theory, not measurement)

Our RTL is designed to accept one 32-bit word per clock whenever `TVALID` and `TREADY` allow it. Since each word contains four pixels, the **best-case steady-state processing rate** at **100 MHz** is:

\[
4\ \mathrm{pixels/clock}\times100\,000\,000\ \mathrm{clocks/s}
=400\,000\,000\ \mathrm{pixels/s}.
\]

For a frame of `W × H` 8-bit grayscale pixels, with dimensions divisible by four:

\[
N_{beats}=\frac{W\times H}{4},\qquad
T_{ideal}=\frac{N_{beats}}{100\,000\,000}.
\]

**What this number means:** the theoretical duration of **N accepted beats**, assuming an uninterrupted stream and ignoring startup/drain cycles. The RTL contains a registered stage, so latency can exceed this ideal beat-count bound even when throughput is one beat/clock.

**What it does NOT include:** allocating buffers, moving data through DDR and AXI DMA, Python calls, CPU cache maintenance, DMA setup, or pauses when `TREADY=0`. Those are measured only in Lab07.3, after the real overlay has been tested.

In [ ]:
clock_hz = 100_000_000  # intended design frequency, verify in Vivado
pixels_per_beat = 4
resolutions = [(64, 64), (256, 256), (512, 512)]
ideal_values = []
print('{:>13}  {:>12}  {:>12}  {:>20}'.format('Image', 'Pixels', 'AXI beats', 'Ideal stream time (us)'))
for h, w in resolutions:
    n_pixels = w*h
    n_beats = (n_pixels + pixels_per_beat-1)//pixels_per_beat
    ideal_us = n_beats/clock_hz * 1e6
    ideal_values.append(ideal_us)
    print('{:>6}×{:<6}  {:>12d}  {:>12d}  {:>20.2f}'.format(w, h, n_pixels, n_beats, ideal_us))

fig, ax = plt.subplots(figsize=(8.5, 4))
labels = ['{} × {}'.format(w, h) for h, w in resolutions]
ax.bar(labels, ideal_values)
for i, v in enumerate(ideal_values):
    ax.text(i, v+max(ideal_values)*.012, '{:.2f} µs'.format(v), ha='center')
ax.set_ylim(0, max(ideal_values)*1.15)
ax.set_xlabel('Frame size (pixels)')
ax.set_ylabel('Ideal stream occupancy (microseconds)')
ax.set_title('Theoretical pixel throughput at 100 MHz, four pixels per beat')
ax.grid(axis='y', alpha=.2)
plt.tight_layout(); plt.show()

### Step 14 — Put theory in context (Lab06 → Lab07)

**Lab06:** one MAC operation could involve several AXI register transactions, so communication overhead could dominate the actual arithmetic. The batch MAC became more efficient by performing many operations per host command.

**Lab07:** we apply the same principle to images: instead of writing each pixel through AXI GPIO, **AXI DMA transfers a complete image as a stream**. A full image is processed in the FPGA one 32-bit beat at a time, potentially **four pixels in parallel**. The total call still includes Python and DMA overhead.

The critical comparison in **Lab07.3** will be:

1. A reference implementation on ARM using Python / NumPy / OpenCV.
2. An **end-to-end FPGA+DMA measurement**, including host setup and transfers.
3. The **ideal throughput calculation** from Step 13, used only as a theoretical reference — *not* a measured FPGA-only timing counter.

There is **no guarantee** that the simple FPGA threshold will outperform the highly optimized vectorized NumPy/OpenCV routines. That is part of the experiment, not a failure.

## Student exercises — verify the ideas, do not just run the cells

1. **Coordinates and ROI:** choose grid columns **C1…C3** and rows **R3…R5**. Before running Step 3, calculate `x0`, `x_stop`, `y0`, `y_stop`, the last included pixels, and the ROI dimensions. Confirm them from the highlighted rectangle. Which object appears in the crop?
2. **Threshold decisions:** for `T=128`, predict the result for pixels 127, 128 and 129. Why does an ordinary `cv2.THRESH_BINARY` call at 128 differ for the middle pixel?
3. **Histogram:** identify why the right-hand background can become white even though it is not a bright geometric object. Which threshold removes most of that background, and what useful objects does it lose?
4. **Four-pixel packing:** write the exact 32-bit input/output words for `[0, 127, 128, 255]` at `T=128` (manual hexadecimal calculation), then verify them using `pack_four`.
5. **AXI protocol:** explain why the second word of an eight-pixel frame has `TLAST=1`, and why `TVALID=1` alone is insufficient for a transfer.
6. **Throughput:** estimate the ideal processing time for a **640 × 480** grayscale frame at 100 MHz and four pixels per beat. State **at least three** reasons why measured DMA wall time will be longer.
7. **Connection with Lab06:** compare sending one pixel per AXI GPIO call to sending a whole image via DMA. Which overhead is avoided and which overhead remains?

### Completion checklist

- [ ] I can read the image axes and derive the ROI bounds **without guessing**.
- [ ] I can explain thresholding, including its equality condition and sensitivity to background noise.
- [ ] I can convert four 8-bit pixels into one 32-bit word and predict the output of the FPGA.
- [ ] I can distinguish AXI-Stream data from `TKEEP`, `TLAST` and handshake control.
- [ ] I know that ideal clock-cycle throughput is **not** measured end-to-end performance.

**Next:** Lab07.2 will load the actual `lab07.bit/.hwh` overlay and use DMA to compare the FPGA-generated mask against `reference_threshold`, pixel by pixel.